# M4.2 · CNN mimarileri (LeNet → ResNet)

Katmanları üst üste koymanın kuralları. Receptive field'ı, çıktı boyutlarını ve residual bağlantının gradyan akışına etkisini elle hesaplıyor, saf Python'da bunları yazıyor, NumPy'da 20 katmanlık düz ve residual ağlarda gradyan normlarını elle geri yayıyor, PyTorch'ta ResNet'in BasicBlock'unu ve küçük bir ResNet'i kuruyor ve en sonda çıkarım için Conv+BatchNorm birleştirmesini (fusion) yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-2-cnn.html](https://m-gonul.github.io/pytorch-lab/m4-2-cnn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_2_cnn.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"conv_out\", lambda: close([need(ns, \"conv_out\")(32, 3, 1, 1), need(ns, \"conv_out\")(32, 3, 2, 1), need(ns, \"conv_out\")(224, 7, 2, 3), need(ns, \"conv_out\")(7, 3, 3, 0)], [32, 16, 112, 2]))\n    R.test(\"receptive_field\", lambda: close([need(ns, \"receptive_field\")(l) for l in ([(3, 1)] * 2, [(3, 1)] * 3, [(3, 1), (3, 2), (3, 1)], [(7, 2), (3, 2), (3, 1), (3, 1)])], [5, 7, 9, 27]))\n    R.test(\"grad_product\", lambda: close([need(ns, \"grad_product\")([0.1] * 10, False), need(ns, \"grad_product\")([0.1] * 10, True), need(ns, \"grad_product\")([0.5, 2.0], False)], [1e-10, 1.1 ** 10, 1.0]))\n\n    def pl():\n        ensure(list(need(ns, \"plan\")(224, [(7, 2, 3), (3, 2, 1), (3, 1, 1), (3, 2, 1)])) == [112, 56, 56, 28], \"ResNet planı\")\n    R.test(\"plan\", pl)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız katman döngüleri (iç içe değil)\", lambda: max_loops(src, 1, func=\"grad_norms\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    D = 12\n    X = rng.standard_normal((5, D))\n    Ws = [rng.standard_normal((D, D)) * 0.6 / np.sqrt(D) for _ in range(4)]\n\n    def num(residual):\n        def f(X0):\n            h = X0\n            for W in Ws:\n                t = np.tanh(h @ W); h = h + t if residual else t\n            return h.sum()\n        g = np.zeros_like(X); hh = 1e-6\n        for i in np.ndindex(X.shape):\n            Xp, Xm = X.copy(), X.copy(); Xp[i] += hh; Xm[i] -= hh\n            g[i] = (f(Xp) - f(Xm)) / (2 * hh)\n        return np.linalg.norm(g)\n\n    def gr():\n        for res in (False, True):\n            n = need(ns, \"grad_norms\")(X, Ws, res)\n            ensure(len(n) == 5, f\"L + 1 = 5 değer bekleniyordu, {len(n)} geldi\")\n            close(n[0], num(res), rtol=1e-5, what=f\"‖g₀‖ (residual={res}) sayısal gradyanla\")\n            close(n[-1], np.sqrt(X.size), what=\"‖g_L‖ = ‖1‖\")\n    R.test(\"‖g₀‖ sayısal gradyanla aynı (düz ve residual)\", gr)\n\n    def comp():\n        D2, L = 64, 20\n        X2 = rng.standard_normal((32, D2))\n        W2 = [rng.standard_normal((D2, D2)) * 0.5 / np.sqrt(D2) for _ in range(L)]\n        p = need(ns, \"grad_norms\")(X2, W2, False); r = need(ns, \"grad_norms\")(X2, W2, True)\n        ensure(p[0] / p[-1] < 1e-3, f\"düz ağda gradyan sönmeli: oran {p[0] / p[-1]:.3g}\")\n        ensure(r[0] / r[-1] > 0.5, f\"residual ağda gradyan korunmalı: oran {r[0] / r[-1]:.3g}\")\n        R.table(\"20 katman, ‖g_l‖ / ‖g_L‖\", [\"katman\", \"düz\", \"residual\"],\n                [[str(l), f\"{p[l] / p[-1]:.2e}\", f\"{r[l] / r[-1]:.3f}\"] for l in (0, 5, 10, 15, 20)])\n    R.test(\"düz ağda sönme, residual'da koruma\", comp)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import need, ensure\n\n\ndef block_params(i, o, s):\n    p = 9 * i * o + 2 * o + 9 * o * o + 2 * o\n    if s != 1 or i != o:\n        p += i * o + 2 * o\n    return p\n\n\ndef check(ns, R):\n    def blk():\n        B = need(ns, \"BasicBlock\", \"sınıf\")\n        a, b = B(16, 16), B(16, 32, 2)\n        ensure(isinstance(a.shortcut, nn.Identity), \"aynı shape'te shortcut nn.Identity olmalı\")\n        ensure(not isinstance(b.shortcut, nn.Identity), \"stride/kanal değişince projeksiyon gerekli\")\n        ensure(sum(p.numel() for p in a.parameters()) == block_params(16, 16, 1), \"BasicBlock(16, 16) parametre sayısı\")\n        ensure(sum(p.numel() for p in b.parameters()) == block_params(16, 32, 2), \"BasicBlock(16, 32, 2) parametre sayısı\")\n        y = b(torch.randn(2, 16, 8, 8))\n        ensure(tuple(y.shape) == (2, 32, 4, 4), f\"çıktı shape {tuple(y.shape)}\")\n        ensure(bool((y >= 0).all()), \"son ReLU eksik\")\n    R.test(\"BasicBlock: shortcut türleri, parametreler, shape\", blk)\n\n    def net():\n        m = need(ns, \"SmallResNet\", \"sınıf\")(10)\n        want = 3 * 9 * 16 + 32 + block_params(16, 16, 1) * 2 + block_params(16, 32, 2) + block_params(32, 32, 1) + block_params(32, 64, 2) + block_params(64, 64, 1) + 64 * 10 + 10\n        got = sum(p.numel() for p in m.parameters())\n        ensure(got == want, f\"parametre sayısı {got}, {want} bekleniyordu\")\n        ensure(tuple(m(torch.randn(2, 3, 32, 32)).shape) == (2, 10), \"çıktı shape (2, 10)\")\n        return f\"{got:,} parametre\"\n    R.test(\"SmallResNet: parametre sayısı ve çıktı\", net)\n\n    def train():\n        g = torch.Generator().manual_seed(0)\n        n = 256\n        y = torch.randint(0, 4, (n,), generator=g)\n        X = torch.randn(n, 3, 16, 16, generator=g) * 0.5\n        for k in range(4):\n            r, c = (k // 2) * 8, (k % 2) * 8\n            X[y == k, :, r:r + 8, c:c + 8] += 1.5\n        torch.manual_seed(0)\n        m = need(ns, \"SmallResNet\", \"sınıf\")(4)\n        opt = torch.optim.Adam(m.parameters(), lr=3e-3)\n        for _ in range(40):\n            idx = torch.randint(0, n, (64,), generator=g)\n            loss = nn.functional.cross_entropy(m(X[idx]), y[idx])\n            opt.zero_grad(); loss.backward(); opt.step()\n        m.eval()\n        with torch.no_grad():\n            acc = (m(X).argmax(1) == y).float().mean().item()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"sentetik 4 sınıf (parlak çeyrek), 40 adım: doğruluk {acc:.3f}\"\n    R.test(\"SmallResNet öğreniyor\", train)\n",
 "e": "import torch\nfrom torch import nn\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n\n    def mk(bias, stride=1):\n        conv = nn.Conv2d(8, 16, 3, stride, 1, bias=bias).double()\n        bn = nn.BatchNorm2d(16).double()\n        with torch.no_grad():\n            bn.running_mean.copy_(torch.randn(16, generator=g, dtype=torch.float64))\n            bn.running_var.copy_(torch.rand(16, generator=g, dtype=torch.float64) + 0.5)\n            bn.weight.copy_(torch.randn(16, generator=g, dtype=torch.float64))\n            bn.bias.copy_(torch.randn(16, generator=g, dtype=torch.float64))\n        conv.eval(); bn.eval()\n        return conv, bn\n\n    def eq():\n        for bias, s in ((False, 1), (True, 2)):\n            conv, bn = mk(bias, s)\n            f = need(ns, \"fuse_conv_bn\")(conv, bn)\n            ensure(isinstance(f, nn.Conv2d) and f.bias is not None, \"bias'lı nn.Conv2d bekleniyordu\")\n            x = torch.randn(3, 8, 10, 10, generator=g, dtype=torch.float64)\n            with torch.no_grad():\n                close(f(x), bn(conv(x)), what=f\"bias={bias}, stride={s}\")\n    R.test(\"fused conv = bn(conv(x)) (eval)\", eq)\n    if \"fuse_conv_bn\" not in ns:\n        return\n    conv, bn = nn.Conv2d(64, 64, 3, 1, 1, bias=False).eval(), nn.BatchNorm2d(64).eval()\n    f = ns[\"fuse_conv_bn\"](conv, bn)\n    x = torch.randn(32, 64, 32, 32, generator=g)\n    with torch.no_grad():\n        t1, t2 = bench(lambda: bn(conv(x)), repeat=3), bench(lambda: f(x), repeat=3)\n    R.table(\"Çıkarım, (32, 64, 32, 32), 3×3 conv\", [\"yöntem\", \"süre\"],\n            [[\"conv → BN (iki geçiş)\", fmt_time(t1)], [\"senin fused conv\", fmt_time(t2)]],\n            note=f\"{t1 / t2:.2f}× hızlı: BN için tüm aktivasyon tensörünü bir kez daha okuyup yazmak gerekmiyor.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_2_cnn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

CNN tarihinin üç fikri:

- **LeNet → VGG:** küçük çekirdekleri üst üste koymak. İki 3×3 katman bir 5×5'in receptive field'ını görür ama $2\cdot9C^2=18C^2$ parametreyle ($25C^2$ yerine) ve araya bir doğrusal olmayanlık daha ekleyerek.
- **Receptive field** katman katman büyür: $r\leftarrow r+(k-1)\,j$, $j\leftarrow j\cdot s$ ($j$: o ana kadarki toplam stride).
- **ResNet:** her blok $y=\text{ReLU}\big(x+F(x)\big)$. Derinlik arttıkça düz ağların *eğitim* hatası bile artıyordu (optimizasyon sorunu). residual bağlantı Jacobian'ı $I+J_F$ yapar; gradyan kimlik yolundan sönmeden geçer.

BasicBlock: `conv3×3 → BN → ReLU → conv3×3 → BN`, artı shortcut. Çözünürlük ya da kanal sayısı değişen blokta shortcut bir `conv1×1 (stride) → BN` projeksiyonudur.

Çıkarımda (eval) BatchNorm sabit bir afin dönüşümdür ve önceki conv'a katlanabilir:

$$
W'=W\cdot\frac{\gamma}{\sqrt{\sigma^2+\epsilon}},\qquad b'=(b-\mu)\frac{\gamma}{\sqrt{\sigma^2+\epsilon}}+\beta .
$$

## 1 · Elle

İlk grupta türev çarpımlarının ne kadar hızlı söndüğünü gör: 10 katmanda $10^{-10}$. Residual ile aynı katmanlar $\approx2.6$ veriyor; gradyan artık kaybolmuyor (pratikte BN ve başlatma büyümeyi de dengeler).

Receptive field'da stride'ın etkisi: stride 2'den sonraki her 3×3 katman receptive field'ı 2 değil 4 piksel büyütüyor. Derin CNN'ler alt örnekleme sayesinde birkaç düzine katmanda tüm görüntüyü görür.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Gradyan akışı: 10 katman, her katmanın yerel türevi $a=0.1$**

- `plain`: Düz: $\prod a$
- `res`: Residual $h+f(h)$: $\prod(1+a)$

**Receptive field (stride 1)**

- `rf2`: İki 3×3 conv
- `rf3`: Üç 3×3 conv
- `ratio`: Üç 3×3 ($C\to C$) / bir 7×7 parametre oranı (bias yok)
- `rfs`: 3×3 s1 → 3×3 s2 → 3×3 s1

**ResNet giriş katmanları, 224×224**

- `stem`: 7×7, stride 2, pad 3 sonrası
- `pool`: ardından 3×3 max-pool, stride 2, pad 1

In [ ]:
cevaplar = dict(
    plain=None,
    res=None,
    rf2=None,
    rf3=None,
    ratio=None,
    rfs=None,
    stem=None,
    pool=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"plain":[1e-10,1e-12],"res":[2.593742,0.001],"rf2":[5,0],"rf3":[7,0],"ratio":[0.55102,0.001],"rfs":[9,0],"stem":[112,0],"pool":[56,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python:

- `conv_out(H, k, s, p)` → çıktı boyutu.
- `receptive_field(layers)` — `layers`: `(k, s)` çiftleri listesi.
- `grad_product(local_derivs, residual)` → zincir kuralı çarpımı; `residual=True` ise her katman $1+a$.
- `plan(H, layers)` — `layers`: `(k, s, p)` listesi → her katmandan sonraki boyutlar listesi.

In [ ]:
%%writefile stage_b.py
def conv_out(H, k, s, p):
    ...


def receptive_field(layers):
    ...


def grad_product(local_derivs, residual):
    ...


def plan(H, layers):
    ...


print(receptive_field([(3, 1), (3, 2), (3, 1)]))        # 9
print(grad_product([0.1] * 10, False), grad_product([0.1] * 10, True))
print(plan(224, [(7, 2, 3), (3, 2, 1)]))                # [112, 56]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Düz ve residual derin ağda gradyanın katmanlar boyunca normunu **elle** geri yay ($D$ genişlik, $L$ katman, tanh):

$$
\text{düz: }h_{l+1}=\tanh(h_lW_l),\qquad \text{residual: }h_{l+1}=h_l+\tanh(h_lW_l),\qquad \mathcal L=\sum h_L .
$$

Geri: $g_L=\mathbf 1$; düz için $g_l=\big(g_{l+1}\odot(1-t^2)\big)W_l^\top$, residual için $g_l=g_{l+1}+\big(g_{l+1}\odot(1-t^2)\big)W_l^\top$ ($t$ o katmanın tanh çıktısı).

- `grad_norms(X, Ws, residual)` → uzunluğu $L+1$ liste: $\lVert g_0\rVert,\dots,\lVert g_L\rVert$ (Frobenius). Yalnız katman döngüleri (ileri ve geri, iç içe değil).

Kontrol 20 katmanda iki ağı karşılaştırır.

In [ ]:
%%writefile stage_c.py
import numpy as np


def grad_norms(X, Ws, residual):
    """→ [‖g_0‖, …, ‖g_L‖]. Yalnız katman döngüleri."""
    ...


rng = np.random.default_rng(0)
D, L = 64, 20
X = rng.standard_normal((32, D))
Ws = [rng.standard_normal((D, D)) * 0.5 / np.sqrt(D) for _ in range(L)]
print(grad_norms(X, Ws, False)[0], grad_norms(X, Ws, True)[0])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `BasicBlock(in_c, out_c, stride=1)` — `nn.Module`; conv'lar `bias=False`, projeksiyon gerekiyorsa `self.shortcut = nn.Sequential(conv1×1, BN)`, gerekmiyorsa `nn.Identity()`. `forward`: `relu(bn2(conv2(relu(bn1(conv1(x))))) + shortcut(x))`.
- `SmallResNet(num_classes)` — stem: `conv3×3 (3→16) + BN + ReLU`; üç aşama, her biri 2 BasicBlock: 16 (stride 1), 32 (stride 2), 64 (stride 2); `AdaptiveAvgPool2d(1)` + `Linear(64, num_classes)`.

Kontrol parametre sayısını, shape'leri, shortcut türlerini ve küçük bir sentetik görevde öğrenmeyi denetler.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        ...

    def forward(self, x):
        ...


class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `fuse_conv_bn(conv, bn)` → yeni bir `nn.Conv2d` (bias'lı); eval modunda `bn(conv(x))` ile aynı çıktı. Conv'un bias'ı yoksa 0 say. **Döngü yok** (kanal başına işlemler broadcasting ile). Parametreleri `torch.no_grad()` altında kopyala.

Kontrol rastgele running istatistiklerle eşitliği ve hız kazancını ölçer.

In [ ]:
%%writefile stage_e.py
import copy
import torch
from torch import nn


@torch.no_grad()
def fuse_conv_bn(conv, bn):
    """→ yeni nn.Conv2d; eval modunda bn(conv(x)) ile aynı. Döngü yok."""
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [He et al. — Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) — arXiv
- [Dive into Deep Learning §8 Modern CNNs (LeNet, VGG, ResNet)](https://d2l.ai/chapter_convolutional-modern/index.html) — CC BY-SA 4.0 / MIT-0
- [UvA DL Tutorial 5: Inception, ResNet and DenseNet](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial5/Inception_ResNet_DenseNet.html) — MIT
- [PyTorch Tutorials — Fusing Convolution and Batch Norm](https://docs.pytorch.org/tutorials/intermediate/custom_function_conv_bn_tutorial.html) — BSD-3